In [1]:
import pandas as pd
import requests
import geopandas as gpd
from rasterstats import zonal_stats
import os, glob
import rasterio
import numpy as np
from rasterio.features import rasterize
from datetime import datetime
from dateutil.relativedelta import relativedelta
import re

In [2]:
def get_key_from_environment(file_path: str, key: str) -> str | None:
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read()

    # Regex to match key: 'value' or key: "value"
    pattern = rf'{key}\s*:\s*[\'"]([^\'"]+)[\'"]'
    match = re.search(pattern, content)

    return match.group(1) if match else None


# Example usage
file_path = "../src/environments/environment.ts"
api_key = get_key_from_environment(file_path, "apiToken")

In [3]:
def get_data(scale):
  header = {
      "Authorization": f"Bearer {api_key}",
      "Content-Type": "application/json"
  }


  start_date = datetime(2021, 4, 1)   
  end_date = datetime(2026, 3, 1)  

  # Loop over months
  date = start_date
  while date <= end_date:
      date_str = date.strftime("%Y-%m")
      url = f"https://api.hcdp.ikewai.org/raster?datatype=spi&period=month&timescale=timescale{scale:03d}&date={date_str}"
      res = requests.get(url, headers=header)
          
      if res.status_code == 200:
          file = f"./data/spi{scale:03d}_{date_str}.tif"
          with open(file, "wb") as f:
              f.write(res.content)
          
      # Move to next month
      date += relativedelta(months=1)
    
get_data(3)

In [ ]:
# Diverging Stacked Area Chart.

In [5]:
thresholds = [
    (-np.inf, -2,   "#730000", "D4"),
    (-2,      -1.6, "#FF0000", "D3"),
    (-1.6,    -1.3, "#FF9900", "D2"),
    (-1.3,    -0.8, "#FFD37F", "D1"),
    (-0.8,    -0.5, "#FFFF00", "D0"),
    (-0.5,     0.5, "#FFFFFF", "Near Normal"),
    ( 0.5,     0.8, "#99CCFF", "W0"),
    ( 0.8,     1.3, "#0066CC", "W1"),
    ( 1.3,     1.6, "#0066CC", "W2"),
    ( 1.6,     2.0, "#003366", "W3"),
    ( 2.0,   np.inf,"#001933", "W4"),
]
labels = [t[3] for t in thresholds]

rows = []

start_date = datetime(2021, 4, 1)   # Sept 2024
end_date = datetime(2026, 3, 1)     # Aug 2025

# Loop over months
date = start_date
while date <= end_date:
  date_str = date.strftime("%Y-%m")
  file = f"./data/spi003_{date_str}.tif"
  with rasterio.open(file) as src:
      data = src.read(1)
  valid = np.isfinite(data)

  total = valid.sum()
  out = {"month": date_str}

  for lo, hi, _, lab in thresholds:
      in_bin = (data >= lo) & (data < hi) & valid
      out[lab] = (in_bin.sum() / total) * 100.0

  rows.append(out)
  date += relativedelta(months=1)

df = pd.DataFrame(rows)
df.to_csv("../public/spi/spi3_distribution_2025.csv", index=False)
df


,month,D4,D3,D2,D1,D0,Near Normal,W0,W1,W2,W3,W4
0,2021-04,0.000000,0.000000,0.000000,0.671567,1.813996,17.731602,28.488140,37.839734,5.763882,4.836744,2.854335
1,2021-05,0.000000,0.000000,0.000000,0.000000,0.371897,21.094301,23.491664,38.008841,8.710931,6.653518,1.668849
2,2021-06,2.423754,4.506863,5.460739,19.744915,15.393617,33.546077,11.279138,5.547550,1.541063,0.548296,0.007987
3,2021-07,7.228552,7.599754,7.296611,7.618853,9.855790,38.800554,12.806659,7.188966,1.058049,0.481973,0.064240
4,2021-08,2.624113,3.427286,3.992250,10.192616,11.378102,46.510732,9.136303,8.891844,2.288330,1.558425,0.000000
5,2021-09,0.200012,0.835466,2.347708,10.970439,7.302862,56.901275,12.566714,5.801037,1.409458,1.246254,0.418775
6,2021-10,0.230222,0.930610,1.937614,14.798790,11.281569,54.993871,9.288743,4.550963,1.227850,0.469819,0.289948
7,2021-11,2.729328,12.308018,17.648958,20.494265,6.817764,38.005716,1.492102,0.503849,0.000000,0.000000,0.000000
8,2021-12,0.097228,0.134383,0.170496,1.553911,4.656178,54.411198,20.962696,14.954355,2.144224,0.849008,0.066323
9,2022-01,0.000000,0.249668,1.482726,14.414393,10.302691,50.465826,11.485053,6.416351,3.331794,1.851498,0.000000
